In [1]:
import glob
import json
import os

base_path = r"C:\Users\Administrator\.cache\kagglehub\competitions\ai12-level1-project"
annotation_root = os.path.join(
    base_path, "sprint_ai_project1_data", "train_annotations"
)

all_json_paths = glob.glob(
    os.path.join(annotation_root, "**", "*.json"), recursive=True
)
print(f"glob으로 찾은 전체 JSON 경로 개수: {len(all_json_paths)}")

merged_images = {}
merged_annotations = []
merged_categories = {}

img_id_counter = 1
ann_id_counter = 1
file_to_img_id = {}
seen_annotations = set()

for json_path in all_json_paths:
    with open(json_path, "r", encoding="utf-8") as fp:
        data = json.load(fp)

    img_info = data["images"][0]
    file_name = img_info["file_name"]
    ann = data["annotations"][0]
    cat = data["categories"][0]

    dedup_key = (file_name, ann["category_id"], tuple(ann["bbox"]))
    if dedup_key in seen_annotations:
        continue
    seen_annotations.add(dedup_key)

    if file_name not in merged_images:
        new_img_id = img_id_counter
        img_id_counter += 1
        file_to_img_id[file_name] = new_img_id
        img_copy = dict(img_info)
        img_copy["id"] = new_img_id
        merged_images[file_name] = img_copy
    else:
        new_img_id = file_to_img_id[file_name]

    ann_copy = dict(ann)
    ann_copy["id"] = ann_id_counter
    ann_copy["image_id"] = new_img_id
    ann_id_counter += 1
    merged_annotations.append(ann_copy)

    merged_categories[cat["id"]] = cat

coco_format = {
    "images": list(merged_images.values()),
    "annotations": merged_annotations,
    "categories": list(merged_categories.values()),
}

print("\n=== 병합 결과 ===")
print(f"images 개수: {len(coco_format['images'])}")
print(f"annotations 개수: {len(coco_format['annotations'])}")
print(f"categories 개수: {len(coco_format['categories'])}")

avg_per_image = len(coco_format["annotations"]) / len(coco_format["images"])
print(f"이미지당 평균 약(bbox) 개수: {avg_per_image:.2f}")

output_path = os.path.join(os.getcwd(), "merged_annotations.json")
with open(output_path, "w", encoding="utf-8") as fp:
    json.dump(coco_format, fp, ensure_ascii=False, indent=2)

print(f"\n병합된 JSON 저장 완료: {output_path}")


glob으로 찾은 전체 JSON 경로 개수: 763

=== 병합 결과 ===
images 개수: 232
annotations 개수: 763
categories 개수: 56
이미지당 평균 약(bbox) 개수: 3.29

병합된 JSON 저장 완료: c:\Users\Administrator\Desktop\python_study\medication-detection\pill_detection_steps\merged_annotations.json
